# 04 — Embeddings

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/04_embeddings.ipynb)

## 📌 What is it?
Embeddings are dense numerical vector representations of text (or images/audio) that capture semantic meaning. Similar meanings produce similar vectors.

## 🧠 Why AI Engineers need this
Embeddings power semantic search, RAG retrieval, clustering, recommendation systems, and are the core mechanism behind how LLMs understand language.

In [ ]:
import numpy as np

# ── WHAT IS AN EMBEDDING? ──
# A word/sentence → dense vector of floats
# Semantically similar texts have HIGH cosine similarity

# Simulated embeddings (real ones come from API or model)
def mock_embed(text: str, dim: int = 8) -> np.ndarray:
    """Mock embedder — real: use OpenAI or sentence-transformers"""
    np.random.seed(hash(text) % 2**31)
    return np.random.randn(dim)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# ── SEMANTIC SIMILARITY ──
texts = [
    "Machine learning is a subset of artificial intelligence",
    "AI and ML are related fields in computer science",     # similar
    "Python is a programming language",                      # different
    "The weather is nice today",                             # very different
]

query = texts[0]
query_emb = mock_embed(query)
print(f"Query: {query[:50]}\n")

for text in texts[1:]:
    emb = mock_embed(text)
    sim = cosine_similarity(query_emb, emb)
    print(f"Sim: {sim:+.3f} | {text[:55]}")

In [ ]:
import numpy as np

# ── REAL EMBEDDING CODE PATTERNS ──
openai_embed_code = '''
from openai import OpenAI
import numpy as np

client = OpenAI()

def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    '''Get embedding from OpenAI API.'''
    text = text.replace("\n", " ")   # clean newlines
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

def get_batch_embeddings(texts: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    '''Embed multiple texts in one API call (efficient!).'''
    texts = [t.replace("\n", " ") for t in texts]
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]
'''

sentence_transformers_code = '''
# LOCAL — no API key needed! (pip install sentence-transformers)
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")   # 384d, fast, free

texts = ["Python is great", "I love machine learning", "The cat sat on the mat"]
embeddings = model.encode(texts)          # (3, 384) numpy array
print(f"Shape: {embeddings.shape}")

# Semantic similarity
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(embeddings)
print(sim_matrix.round(3))
'''

print("OpenAI embeddings API:")
print(openai_embed_code)
print("\nLocal embeddings (free, no API):")
print(sentence_transformers_code)

In [ ]:
import numpy as np

# ── SEMANTIC SEARCH ──
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_search(query_emb: np.ndarray,
                    corpus_embs: np.ndarray,
                    corpus_texts: list[str],
                    top_k: int = 3) -> list[dict]:
    """Find top-k most similar documents to the query."""
    similarities = [cosine_similarity(query_emb, doc_emb)
                   for doc_emb in corpus_embs]
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    return [
        {"text": corpus_texts[i], "score": similarities[i]}
        for i in top_indices
    ]

# Mock corpus
np.random.seed(42)
corpus = [
    "Python generators use yield to produce values lazily",
    "Decorators modify function behavior using @syntax",
    "RAG retrieves relevant documents before LLM generation",
    "Embeddings are dense vector representations of text",
    "FAISS enables fast approximate nearest neighbor search",
]

# Simulate embeddings (use real model in production)
dim = 32
corpus_embs = np.array([np.random.randn(dim) for _ in corpus])
query_emb = corpus_embs[2] + np.random.randn(dim) * 0.2  # close to RAG doc

results = semantic_search(query_emb, corpus_embs, corpus, top_k=3)
print("Query: 'How does RAG work?'\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['score']:.3f}] {r['text']}")

## 🔗 What's Next?
[05 — Vector Databases →](05_vector_databases.ipynb)